# מחברת 2: patterns של kernels

כאן מריצים שלושה kernels ומקשרים כל אחד לרעיון אחר: grid-stride loop, shared-memory reduction ו-2D tiling.

In [ ]:
from pathlib import Path
import subprocess

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the cuda-kernels-a100-beginners repository")

ROOT = find_repo_root(Path.cwd())
subprocess.run(["cmake", "-S", str(ROOT), "-B", str(ROOT / "build"), "-DCMAKE_CUDA_ARCHITECTURES=80"], check=True)
subprocess.run(["cmake", "--build", str(ROOT / "build"), "-j"], check=True)


In [ ]:
targets = ["07_grid_stride", "08_reduction", "09_tiled_matmul"]
outputs = {}
for target in targets:
    result = subprocess.run([str(ROOT / "build" / target)], text=True, capture_output=True)
    outputs[target] = result.stdout.strip()
    print(target, "->", result.stdout.strip())
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0
    assert "PASS" in result.stdout


## קריאת קוד ממוקדת

התא הבא מדפיס רק שורות עם primitives חשובים. לאחר מכן פתחו את הקבצים המלאים בעורך.

In [ ]:
needles = ("gridDim", "__shared__", "__syncthreads", "dim3", "blockIdx")
for target in targets:
    path = ROOT / "lessons" / f"{target}.cu"
    print(f"\n== {path.name} ==")
    for number, line in enumerate(path.read_text().splitlines(), 1):
        if any(needle in line for needle in needles):
            print(f"{number:3}: {line}")


## ניסוי

בחרו שינוי אחד בלבד בכל פעם:

- מספר blocks ב-grid-stride loop
- מספר threads ב-reduction
- גודל `TILE` ב-matmul

לאחר כל שינוי: בנו, בדקו `PASS`, ורק אז מדדו.